PDF Chatbot using LLamaIndex

In [4]:
# !pip install pypdf
# !pip install chromadb llama-index llama-index-vector-stores-chroma llama-index-llms-ollama llama-index-embeddings-ollama

In [3]:
import requests
import chromadb
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext, Settings
from llama_index.vector_stores.chroma import ChromaVectorStore
import threading
import os, subprocess
import time
from pathlib import Path
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding
from IPython.display import display, Markdown
import pypdf

In [5]:
# !sudo apt update
# !sudo apt install -y pciutils
# !sudo apt-get install zstd
# !curl -fsSL https://ollama.com/install.sh | sh

In [6]:
def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

In [7]:
!ollama pull llama3.2
!ollama pull nomic-embed-text

In [8]:
# Configure the LLM to use Ollama with the llama3.2 model and increase the timeout
Settings.llm = Ollama(model="llama3.2", request_timeout=360.0)

# Configure the embedding model to use Ollama with the nomic-embed-text model
Settings.embed_model = OllamaEmbedding(model_name="nomic-embed-text")

print("LlamaIndex settings updated to use Ollama for LLM (llama3.2 with increased timeout) and embedding (nomic-embed-text).")

LlamaIndex settings updated to use Ollama for LLM (llama3.2 with increased timeout) and embedding (nomic-embed-text).


In [9]:
# Get the pdf
url = "https://www.nrb.org.np/contents/uploads/2026/03/Macroeconomic-Report-February-2026.pdf"
Path("data").mkdir(exist_ok=True)
local_filename = "./data/Macroeconomic-Report-February-2026.pdf"

# Send a GET request to the URL
response = requests.get(url)

# Check if the request was successful (Status Code 200)
if response.status_code == 200:
    # Open a local file in 'wb' (write binary) mode and save the content
    with open(local_filename, "wb") as file:
        file.write(response.content)
    print("Download complete!")
else:
    print(f"Failed to download. Status code: {response.status_code}")

text_output_path = "./data/Macroeconomic-Report-February-2026.txt"

extracted_text = ""
with open(local_filename, 'rb') as file:
    reader = pypdf.PdfReader(file)
    for page_num in range(len(reader.pages)):
        page = reader.pages[page_num]
        extracted_text += page.extract_text() + "\n"

# Save the extracted text to a .txt file
with open(text_output_path, 'w', encoding='utf-8') as f:
    f.write(extracted_text)

print(f"Text extracted from PDF and saved to {text_output_path}")

# delete the pdf
if os.path.exists(local_filename):
    os.remove(local_filename)
    print("File deleted successfully.")
else:
    print("The file does not exist.")

Download complete!
Text extracted from PDF and saved to ./data/Macroeconomic-Report-February-2026.txt
File deleted successfully.


In [10]:
# Load data
documents = SimpleDirectoryReader("./data").load_data()

In [11]:
# initialize client, setting path to save data
db = chromadb.PersistentClient(path="./chroma_db")

In [12]:
# create collection
chroma_collection = db.get_or_create_collection("rag_data_collection")

In [13]:
# assign chroma as the vector_store to the context
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [14]:
# create your index with a progress bar
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
    show_progress=True
)

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/40 [00:00<?, ?it/s]

In [17]:
from llama_index.core import set_global_handler
set_global_handler("simple")

# create a query engine and query
query_engine = index.as_query_engine()
response = query_engine.query("How is remittance contributing to the economy?")
# print(response)
display(Markdown(response.response))

Remittances are playing a dominant role in offsetting the country's widening trade deficit. They have increased significantly over the years, from negligible levels in 2010 to 28.2 percent of GDP by 2025. This substantial inflow of remittances is contributing to the country's current account and overall balance of payments (BoP) remaining mostly in surplus, particularly after the 2000s.

In [18]:
# Display the source nodes for references
print("\n--- Source Nodes (References) ---")
for i, node in enumerate(response.source_nodes):
    print(f"Source Node {i+1}:\n")
    print(node.get_content().strip())
    print("\n---------------------------------")


--- Source Nodes (References) ---
Source Node 1:

Net exports, 
which was just -6.6 percent of GDP in 1980, 
has deteriorated sharply over time reaching 
around -25% by 2025 (Table 3.4). This 
deterioration in the trade balance indicates 
that Nepal experienced a persistent and 
widening trade deficit.  However, despite 
this worsening trade position, the current 
account and overall BoP remains mostly in 
surplus, especially after 2000s, reflecting 
the rising role of remittances as the 
dominant offsetting inflow. Remittances 
increased from negligible levels to 19.4 
percent of GDP in 2010 and further to 28.2 
MACROECONOMIC REPORT | FEBRUARY 2026
37
percent by 2025, more than compensation 
for the larger trade deficit and even turning 
the current account positive in most of 
the years. This pattern indicates that 
Nepal’s external balance is driven more by 
transfers, primarily remittances than by 
exports of goods and services, making the 
BoP outcomes heavily reliant on sustaine